# Week 10 — Fashion Image Classifier with a CNN

**Theme:** Convolutional neural networks — theory & image classification practice

Last week's network flattened every image into a flat list of 784 numbers —
which throws away the fact that nearby pixels are related. A
**Convolutional Neural Network (CNN)** instead slides small filters across the
image, learning to detect local patterns (edges, corners, textures) that get
combined into more complex shapes in deeper layers.

**Dataset:** FashionMNIST — 70,000 grayscale images (28x28) of clothing items
across 10 categories (t-shirt, trouser, sneaker, bag, ...). Same shape as
MNIST, but visually a much harder task.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
import torchvision
import torchvision.transforms as transforms
import matplotlib.pyplot as plt

torch.manual_seed(0)
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)

class_names = ["T-shirt/top", "Trouser", "Pullover", "Dress", "Coat",
               "Sandal", "Shirt", "Sneaker", "Bag", "Ankle boot"]

In [ ]:
transform = transforms.ToTensor()
train_data = torchvision.datasets.FashionMNIST(root="./data", train=True, download=True, transform=transform)
test_data = torchvision.datasets.FashionMNIST(root="./data", train=False, download=True, transform=transform)

train_loader = DataLoader(train_data, batch_size=64, shuffle=True)
test_loader = DataLoader(test_data, batch_size=256, shuffle=False)

images, labels = next(iter(train_loader))
fig, axes = plt.subplots(2, 5, figsize=(9, 4))
for i, ax in enumerate(axes.flat):
    ax.imshow(images[i][0], cmap="gray_r")
    ax.set_title(class_names[labels[i]], fontsize=9)
    ax.axis("off")
plt.suptitle("A batch of FashionMNIST images")
plt.tight_layout()
plt.show()

## 1. What does a convolution do?

A `Conv2d` layer slides a small learnable filter (e.g. 3x3) across the image
and computes a weighted sum at each position — the result is a "feature map"
that lights up wherever the filter's pattern appears. `MaxPool2d` then shrinks
the feature map by keeping only the strongest response in each small region,
making the network robust to small shifts in position.

In [ ]:
class SimpleCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(1, 16, kernel_size=3, padding=1)   # 28x28 -> 28x28, 16 filters
        self.conv2 = nn.Conv2d(16, 32, kernel_size=3, padding=1)  # 14x14 -> 14x14, 32 filters
        self.pool = nn.MaxPool2d(2, 2)                            # halves spatial size
        self.fc1 = nn.Linear(32 * 7 * 7, 64)
        self.fc2 = nn.Linear(64, 10)

    def forward(self, x):
        x = self.pool(F.relu(self.conv1(x)))   # -> (16, 14, 14)
        x = self.pool(F.relu(self.conv2(x)))   # -> (32, 7, 7)
        x = x.view(x.size(0), -1)              # flatten for the classifier head
        x = F.relu(self.fc1(x))
        x = self.fc2(x)
        return x

cnn = SimpleCNN().to(device)
print("CNN parameters:", sum(p.numel() for p in cnn.parameters()))

## 2. Compare to last week's plain feedforward network

Same task, same dataset, roughly comparable training budget — does spatial
structure actually help?

In [ ]:
class SimpleMLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(28 * 28, 128)
        self.fc2 = nn.Linear(128, 10)

    def forward(self, x):
        x = x.view(x.size(0), -1)
        x = F.relu(self.fc1(x))
        return self.fc2(x)

mlp = SimpleMLP().to(device)
print("MLP parameters:", sum(p.numel() for p in mlp.parameters()))

In [ ]:
def evaluate(model, loader):
    model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            preds = model(images).argmax(dim=1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)
    return correct / total

def train(model, n_epochs=3, lr=1e-3):
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    test_accs = []
    for epoch in range(n_epochs):
        model.train()
        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)
            optimizer.zero_grad()
            loss = criterion(model(images), labels)
            loss.backward()
            optimizer.step()
        acc = evaluate(model, test_loader)
        test_accs.append(acc)
        print(f"  epoch {epoch+1}: test_acc={acc:.2%}")
    return test_accs

print("Training MLP...")
mlp_accs = train(mlp)
print("Training CNN...")
cnn_accs = train(cnn)

In [ ]:
plt.figure(figsize=(6, 4))
plt.plot(range(1, len(mlp_accs) + 1), mlp_accs, marker="o", label="Feedforward (MLP)")
plt.plot(range(1, len(cnn_accs) + 1), cnn_accs, marker="o", label="CNN")
plt.title("Test Accuracy: MLP vs. CNN on FashionMNIST")
plt.xlabel("Epoch")
plt.ylabel("Test accuracy")
plt.legend()
plt.ylim(0, 1)
plt.show()

## 3. Look inside the CNN: filters and feature maps

The first convolutional layer's filters are small 3x3 patterns — we can view
them directly. Then we can see what each one "lights up" on a real image.

In [ ]:
filters = cnn.conv1.weight.data.cpu()  # shape: (16, 1, 3, 3)
fig, axes = plt.subplots(2, 8, figsize=(10, 3))
for i, ax in enumerate(axes.flat):
    ax.imshow(filters[i, 0], cmap="gray")
    ax.axis("off")
plt.suptitle("Learned 3x3 Filters in conv1 (16 filters)")
plt.show()

In [ ]:
sample_image = images[0:1].to(device)  # one FashionMNIST image
with torch.no_grad():
    feature_maps = F.relu(cnn.conv1(sample_image))[0].cpu()  # shape: (16, 28, 28)

fig, axes = plt.subplots(2, 8, figsize=(10, 3))
for i, ax in enumerate(axes.flat):
    ax.imshow(feature_maps[i], cmap="viridis")
    ax.axis("off")
plt.suptitle(f"Feature Maps for a '{class_names[labels[0]]}' Image")
plt.show()

## Try it yourself

1. **Add a third conv layer.** Insert `conv3 = nn.Conv2d(32, 64, 3, padding=1)`
   with another pool — remember to update `fc1`'s input size accordingly.
2. **Bigger filters.** Try `kernel_size=5` instead of 3 in `conv1` — does it
   help or hurt?
3. **Confusion matrix.** Using `sklearn.metrics.confusion_matrix` (as in
   Week 7), find which two FashionMNIST classes the CNN confuses most (hint:
   "shirt" vs. "T-shirt/top" and "pullover" vs. "coat" are usually hard).
4. **Parameter efficiency.** The CNN and MLP have different parameter counts —
   which one gets higher accuracy *per parameter*? What does that tell you
   about why CNNs are the default choice for images?